# Profiling the AutoScout24 snapshot for the dataset card

This notebook is the exploratory analysis behind the [dataset card](../docs/docs/dataset-card.md).
It recomputes the card's figures from the pinned Zenodo file, so a reader can check the card instead of trusting it.

It runs top to bottom on a fresh kernel in the project environment (`uv run jupyter lab`).
It needs the raw CSV in `data/external/`, which the `download` stage caches there; if the file is missing, the notebook fetches it from Zenodo the same way the stage does, and it refuses to read bytes that do not hash to the pinned MD5.

There is one section per card section, in the card's order.
Each section ends with a table of checks: the claim, the value the card prints, the value computed here, and whether the two agree at the precision the card prints.
The summary at the end lists every check in one table, and fails when a figure does not reproduce, so running the notebook checks the card as well as the code.

In [ ]:
import re
import unicodedata

from IPython.display import display
import numpy as np
import pandas as pd

from recommenditos.config import EXTERNAL_DATA_DIR, PARAMS_FILE
from recommenditos.data.build_features import weight_kg
from recommenditos.data.download_raw_dataset import raw_csv_path, read_raw_csv
from recommenditos.data.preprocess import apply_row_rules, derive_group_key_and_drop_pii
from recommenditos.data.split_data import (
    SPLIT_NAMES,
    assign_split,
    cross_holdout_sellers,
    realised_shares,
    supported_makes,
)
from recommenditos.pipeline import load_params
from recommenditos.schema import RAW_SCHEMA

pd.set_option("display.max_rows", 500)
pd.set_option("display.max_colwidth", 120)

## How a card figure is checked

`check` puts one card figure next to the value computed for it.
The card rounds percentages and sizes to one decimal, so the computed value is rounded to the card's precision before the two are compared; counts, dates and names must be equal exactly.
Each section collects its checks into one table with `checks`, and the summary concatenates those tables, so re-running a cell never counts a check twice.

In [ ]:
CHECK_COLUMNS = ["section", "claim", "card", "computed", "match"]


def pct(part: float, whole: float) -> float:
    """`part` as a percentage of `whole`."""
    return 100 * part / whole


def check(claim: str, card: object, computed: object, decimals: int | None = None) -> dict:
    """One card figure next to the value computed for it, rounded to `decimals` if given."""
    if decimals is not None:
        computed = round(float(computed), decimals)
    elif isinstance(computed, np.generic):
        computed = computed.item()
    if isinstance(card, int) and isinstance(computed, float) and computed.is_integer():
        computed = int(computed)
    # `True == 1` in Python, so a flag and a count must not pass for each other.
    match = isinstance(card, bool) == isinstance(computed, bool) and computed == card
    return {"claim": claim, "card": card, "computed": computed, "match": match}


def checks(section: str, rows: list[dict]) -> pd.DataFrame:
    """One section's checks as a table, labelled with the card section they reproduce.

    The values keep their own types, so a count is never displayed as a float.
    """
    table = pd.DataFrame(rows, dtype=object).assign(section=section)
    return table.astype({"match": bool})[CHECK_COLUMNS]

## The pinned file

Before anything else, the notebook locates the raw CSV and verifies it, using the `download` stage's own functions and the pin in `params.yaml`.
`raw_csv_path` reuses the cached file in `data/external/` or downloads it, and raises unless its MD5 equals `download.md5`.
`read_raw_csv` reads it with the raw contract's dtypes and fails if a column was added or removed upstream.
Conforming the result to `RAW_SCHEMA` gives exactly the frame the stage writes to `data/raw/listings.parquet`.

In [ ]:
params = load_params(PARAMS_FILE)
download = params["download"]
csv_path = raw_csv_path(
    EXTERNAL_DATA_DIR,
    record=download["zenodo_record"],
    filename=download["filename"],
    expected_md5=download["md5"],
)
print(f"{csv_path.name}: MD5 {download['md5']}, verified")

In [ ]:
raw = RAW_SCHEMA.conform(read_raw_csv(csv_path))
raw.shape

## Card header and Dataset Details

The header states the size of the snapshot, and "Dataset Sources" and "Provenance of the figures" identify the file by size and checksum.
The snapshot date is not a column: it is taken from the file name, and the pipeline uses it as `reference_date`.

In [ ]:
snapshot = pd.Timestamp(params["reference_date"])
date_in_name = pd.to_datetime(re.search(r"\d{8}", csv_path.name).group(), format="%Y%m%d")

details_checks = checks(
    "Header and Dataset Details",
    [
        check("listings", 118_382, len(raw)),
        check("columns", 75, raw.shape[1]),
        check("file size (MB)", 548.6, csv_path.stat().st_size / 1_000_000, 1),
        check(
            "MD5 the file was verified against",
            "b23a122cc51baf7de39f449193ff0d28",
            download["md5"],
        ),
        check("snapshot date, from the file name", "2025-11-08", str(date_in_name.date())),
        check("reference_date is the snapshot date", True, date_in_name == snapshot),
    ],
)
details_checks

## Uses

"Out-of-Scope Use" rules out claims about the European market as a whole, because four premium brands dominate the file.

In [ ]:
PREMIUM_MAKES = ["BMW", "Porsche", "Mercedes-Benz", "Audi"]
make_counts = raw["make"].value_counts()
premium_share = pct(make_counts[PREMIUM_MAKES].sum(), len(raw))

uses_checks = checks(
    "Uses", [check("share of the four premium makes (%)", 82.9, premium_share, 1)]
)
uses_checks

## Dataset Structure

The format line and the "Columns" table.
The table assigns every column to one group, so the check is that its groups name each of the file's columns exactly once.

In [ ]:
CARD_COLUMN_GROUPS = {
    "Identifiers": "id",
    "Free text": "description",
    "Ratings": "ratings_average ratings_count ratings_recommend_percentage",
    "Pricing": "price_currency price price_tax_deductible price_negotiable price_net "
    "price_vat_rate",
    "Vehicle identity and body": "vin make model model_version german_hsn_tsn mileage_km_raw "
    "mileage_km registration_date production_year vehicle_type body_type nr_seats nr_doors "
    "body_color paint_type body_color_original upholstery upholstery_color",
    "Drivetrain and mechanicals": "power_kw power_hp transmission gears drive_train cylinders "
    "cylinders_volume_cc weight_kg",
    "Fuel, energy and emissions": "has_particle_filter fuel_category primary_fuel "
    "electric_range_km electric_range_city_km fuel_cons_comb_l100_km fuel_cons_city_l100_km "
    "fuel_cons_highway_l100_km co2_emission_grper_km fuel_cons_comb_l100_wltp_km "
    "fuel_cons_electric_comb_l100_wltp_km co2_emission_grper_wltp_km",
    "Equipment (free-text lists)": "equipment_comfort equipment_entertainment equipment_extra "
    "equipment_safety",
    "Condition flags": "is_used is_new is_preregistered had_accident has_full_service_history "
    "non_smoking nr_prev_owners is_rental",
    "Miscellaneous": "envir_standard original_market offer_type",
    "Location": "country_code zip city street latitude longitude",
    "Seller": "seller_is_dealer seller_type seller_company_name has_warranty warranty",
}

In [ ]:
grouped = [column for names in CARD_COLUMN_GROUPS.values() for column in names.split()]

structure_checks = checks(
    "Dataset Structure",
    [
        check("rows", 118_382, len(raw)),
        check("columns", 75, raw.shape[1]),
        check("currency of every row", "EUR", ", ".join(raw["price_currency"].unique())),
        check("the column groups name every column", True, set(grouped) == set(raw.columns)),
        check("no column is in two groups", True, len(grouped) == len(set(grouped))),
        check("description filled (%)", 96.3, pct(raw["description"].notna().sum(), len(raw)), 1),
    ],
)
structure_checks

### Distribution

The country table, the make table and the list of other key distributions.
The make table names the seven largest makes and folds the rest into one row.

In [ ]:
CARD_COUNTRIES = {
    "DE": 45_611,
    "IT": 23_957,
    "NL": 17_059,
    "BE": 9_582,
    "ES": 8_015,
    "AT": 7_213,
    "FR": 6_141,
    "LU": 789,
    "missing": 15,
}
countries = raw["country_code"].fillna("missing").value_counts()

country_checks = checks(
    "Distribution: countries",
    [
        check(f"{code} listings", listings, countries.get(code, 0))
        for code, listings in CARD_COUNTRIES.items()
    ]
    + [check("no other country occurs", True, set(countries.index) == set(CARD_COUNTRIES))],
)
country_checks

In [ ]:
CARD_MAKES = {
    "BMW": (37_745, 31.9),
    "Porsche": (25_511, 21.5),
    "Mercedes-Benz": (19_400, 16.4),
    "Audi": (15_469, 13.1),
    "Alfa Romeo": (7_806, 6.6),
    "Suzuki": (4_592, 3.9),
    "Volvo": (3_643, 3.1),
}
others = make_counts.iloc[len(CARD_MAKES) :]

make_rows = [check("distinct makes", 25, len(make_counts))]
for make, (listings, share) in CARD_MAKES.items():
    make_rows.append(check(f"{make} listings", listings, make_counts.get(make, 0)))
    make_rows.append(check(f"{make} share (%)", share, pct(make_counts.get(make, 0), len(raw)), 1))
make_rows += [
    check(
        "the table names the largest makes",
        True,
        list(make_counts.index[: len(CARD_MAKES)]) == list(CARD_MAKES),
    ),
    check("makes in 'all others'", 18, len(others)),
    check("'all others' listings", 4_216, others.sum()),
    check("'all others' share (%)", 3.6, pct(others.sum(), len(raw)), 1),
]
make_checks = checks("Distribution: makes", make_rows)
make_checks

In [ ]:
offers = raw["offer_type"].value_counts()
vehicles = raw["vehicle_type"].value_counts()
sellers = raw["seller_type"].value_counts()
registered = pd.to_datetime(raw["registration_date"])
used_offer = raw["offer_type"] == "U"
in_snapshot_year = registered.dt.year == snapshot.year

count_checks = checks(
    "Distribution: other",
    [
        check("offer_type U", 114_127, offers["U"]),
        check("offer_type N", 4_252, offers["N"]),
        check("offer_type A", 3, offers["A"]),
        check(
            "U rows flagged is_preregistered", 3_702, (used_offer & raw["is_preregistered"]).sum()
        ),
        check(f"first registered in {snapshot.year}", 32_199, in_snapshot_year.sum()),
        check("vehicle_type Car", 117_926, vehicles["Car"]),
        check("vehicle_type Transporter", 456, vehicles["Transporter"]),
        check("dealer listings", 98_565, sellers["Dealer"]),
        check("private listings", 19_802, sellers["PrivateSeller"]),
        check("distinct seller_company_name", 17_141, raw["seller_company_name"].nunique()),
    ],
)
count_checks

In [ ]:
CHEAP_EUR = 100
price, mileage = raw["price"], raw["mileage_km_raw"]

range_checks = checks(
    "Distribution: other",
    [
        check("lowest price (EUR)", 1, price.min()),
        check("highest price (EUR)", 13_500_000, price.max()),
        check("median price (EUR)", 39_980, price.median()),
        check(f"rows priced below {CHEAP_EUR} EUR", 8, (price < CHEAP_EUR).sum()),
        check("lowest mileage_km_raw (km)", 0, mileage.min()),
        check("highest mileage_km_raw (km)", 2_570_000, mileage.max()),
        check("median mileage_km_raw (km)", 56_400, mileage.median()),
        check("latest registration_date", "2026-11-01", str(registered.max().date())),
    ],
)
range_checks

### Completeness

A cell counts as filled when it is not missing.
That is the right measure only if no text cell is blank instead of missing, so the cell checks that first.
The `equipment_*` lists count as filled when they are `'[]'`, which is how the file writes an empty list.

In [ ]:
CARD_FILL_RATES = {
    "warranty": 0.0,
    "has_warranty": 0.0,
    "fuel_cons_city_l100_km": 0.0,
    "fuel_cons_highway_l100_km": 0.0,
    "electric_range_city_km": 0.5,
    "fuel_cons_electric_comb_l100_wltp_km": 4.2,
    "electric_range_km": 10.8,
    "production_year": 18.8,
    "co2_emission_grper_km": 21.0,
    "fuel_cons_comb_l100_wltp_km": 25.6,
    "price_vat_rate": 25.7,
    "german_hsn_tsn": 26.0,
    "price_net": 28.7,
    "vin": 34.0,
    "fuel_cons_comb_l100_km": 35.3,
    "original_market": 38.5,
    "co2_emission_grper_wltp_km": 39.2,
    "primary_fuel": 50.7,
    "nr_prev_owners": 54.7,
}

In [ ]:
text_columns = raw.select_dtypes("str").columns
blank_cells = sum(int(raw[column].str.strip().eq("").sum()) for column in text_columns)
filled = raw.notna().mean() * 100
left_out = filled.drop(list(CARD_FILL_RATES)).sort_values()

print(f"blank text cells: {blank_cells}")
print(
    f"sparsest column the table leaves out: {left_out.index[0]}, {left_out.iloc[0]:.1f} % filled"
)
completeness_checks = checks(
    "Completeness",
    [check("fully populated columns", 22, raw.notna().all().sum())]
    + [
        check(f"{column} filled (%)", share, filled[column], 1)
        for column, share in CARD_FILL_RATES.items()
    ],
)
completeness_checks

### Known data issues

The issues in the card's order: the empty columns and the near-constant `had_accident`, the contradicting condition flags, the registrations after the snapshot and the duplicates.

In [ ]:
used, new, preregistered = raw["is_used"], raw["is_new"], raw["is_preregistered"]
after_snapshot = registered > snapshot

issue_checks = checks(
    "Known data issues",
    [
        check(
            "completely empty columns",
            "fuel_cons_city_l100_km, fuel_cons_highway_l100_km, has_warranty, warranty",
            ", ".join(sorted(raw.columns[raw.isna().all()])),
        ),
        check("had_accident True", 3, raw["had_accident"].sum()),
        check(
            "rows neither used, new nor pre-registered",
            14_744,
            (~used & ~new & ~preregistered).sum(),
        ),
        check("offer_type U rows", 114_127, used_offer.sum()),
        check("of those, is_used False", 18_446, (used_offer & ~used).sum()),
        check("registered after the snapshot date", 164, after_snapshot.sum()),
        check("latest registration_date", "2026-11-01", str(registered.max().date())),
        check(
            "of those, registered in January 2026",
            137,
            (registered.dt.strftime("%Y-%m") == "2026-01").sum(),
        ),
    ],
)
issue_checks

The duplicate key is the one the `preprocess` stage deduplicates on, read from `params.yaml`, so the check also confirms that the stage's key is the one the card describes.
A full-row duplicate compares every column, `id` included.

In [ ]:
CARD_DEDUP_KEY = [
    "make",
    "model",
    "model_version",
    "mileage_km_raw",
    "registration_date",
    "price",
    "power_kw",
]
dedup_key = params["preprocess"]["dedup_key"]
same_but_id = raw.drop(columns="id").duplicated().sum()

duplicate_checks = checks(
    "Known data issues",
    [
        check("vin filled (%)", 34.0, filled["vin"], 1),
        check("the stage's dedup key is the card's key", CARD_DEDUP_KEY, dedup_key),
        check("duplicate rows on that key", 6_347, raw.duplicated(subset=dedup_key).sum()),
        check("exact full-row duplicates", 0, raw.duplicated().sum()),
        check("rows identical to an earlier row except for id", 206, same_but_id),
    ],
)
duplicate_checks

The card states the share of all rows whose `description` contains the exact listing price.
A price can be written with or without a thousands separator, so a price of 39980 counts as `39980`, `39.980`, `39,980` or `39 980`.
Two ways of matching are computed.
A plain substring match also counts a price that is only a fragment of a longer number, such as a price of 4000 inside a mileage written `400000km`.
A whole-number match requires that the price is not part of a longer number, which is what "the exact listing price" means, so it is the figure the card states.
The substring match is printed next to it to show how much the looser method overcounts.

In [ ]:
# The thousands separators the descriptions use: comma, dot, space, apostrophe and the
# no-break and narrow no-break spaces.
SEPARATORS = ",. '\u00a0\u202f"
# Next to a number, a space is a word break as often as a separator, so only these
# mark a price as part of a longer number.
BOUNDARY = "[,.']"
CURRENCY_AMOUNT = r"(?:€|EUR|eur|Euro|euro)\s?\d|\d\s?(?:€|EUR|eur|Euro|euro)"


def price_spellings(price: float) -> set[str]:
    """`price` as an integer, with no separator and with each thousands separator."""
    grouped = f"{int(price):,}"
    return {str(int(price))} | {grouped.replace(",", separator) for separator in SEPARATORS}


def mentions_price(text: str, price: float, *, whole_number: bool) -> bool:
    """Whether `text` contains `price`, as a substring or as a whole number.

    A whole number is not preceded by a digit, or a digit and a comma, dot or
    apostrophe, and is not followed by a digit or by one of those and three digits.
    """
    spellings = sorted(price_spellings(price), key=len, reverse=True)
    if not whole_number:
        return any(spelling in text for spelling in spellings)
    alternatives = "|".join(map(re.escape, spellings))
    bounded = rf"(?<!\d)(?<!\d{BOUNDARY})(?:{alternatives})(?!\d)(?!{BOUNDARY}\d{{3}}(?!\d))"
    return re.search(bounded, text) is not None

In [ ]:
described = raw.loc[raw["description"].notna(), ["description", "price"]]
as_substring = sum(
    mentions_price(text, price, whole_number=False)
    for text, price in described.itertuples(index=False)
)
as_whole_number = sum(
    mentions_price(text, price, whole_number=True)
    for text, price in described.itertuples(index=False)
)
currency_amount = described["description"].str.contains(CURRENCY_AMOUNT).sum()

print(f"price as a whole number: {as_whole_number:,} rows; as a substring: {as_substring:,} rows")
print(f"a currency token next to a number: {pct(currency_amount, len(raw)):.1f} % of all rows")
leak_checks = checks(
    "Known data issues",
    [
        check(
            "rows with the exact price in description (%)", 6.8, pct(as_whole_number, len(raw)), 1
        ),
        check(
            "many more rows contain some currency amount", True, currency_amount > as_whole_number
        ),
    ],
)
leak_checks

The redundant column pairs.
`mileage_km` and `power_hp` are checked value by value; `body_color_original` is free text and is not checked.
For `primary_fuel`, the card states how many of its levels occur under more than one `fuel_category`, which is why it overlaps that column rather than refining it.

In [ ]:
KW_TO_HP = 1.35962
km_text = raw["mileage_km"]
has_mileage = km_text.notna() & mileage.notna()
km_parsed = pd.to_numeric(km_text[has_mileage].str.replace(r"\D", "", regex=True))
has_power = raw["power_kw"].notna() & raw["power_hp"].notna()
kw_in_hp = (raw.loc[has_power, "power_kw"] * KW_TO_HP).round()
categories_per_fuel = raw.groupby("primary_fuel")["fuel_category"].nunique()
spread = categories_per_fuel[categories_per_fuel > 1].to_dict()

same_rows = (km_text.notna() == mileage.notna()).all()
same_mileage = (km_parsed == mileage[has_mileage]).all()
same_power = (kw_in_hp == raw.loc[has_power, "power_hp"]).all()
print(f"primary_fuel levels under more than one fuel category: {spread}")
redundancy_checks = checks(
    "Known data issues",
    [
        check("mileage_km is filled exactly where mileage_km_raw is", True, same_rows),
        check("mileage_km is mileage_km_raw as text", True, same_mileage),
        check("power_hp is power_kw in hp, rounded", True, same_power),
        check("primary_fuel levels", 14, categories_per_fuel.size),
        check("primary_fuel levels under more than one fuel_category", 9, len(spread)),
    ],
)
redundancy_checks

The outliers.
The `weight_kg` figures are measured "over the scoped listings", which are the rows the `preprocess` stage keeps.
The cell runs the stage's own steps on the raw frame, the seller key and PII drop and then the row rules in order, and parses `weight_kg` with the `features` stage's parser.
The funnel it returns is reused in "How this project uses the dataset".

In [ ]:
LIGHT_KG, HEAVY_KG = 500, 4_000
scoped, funnel = apply_row_rules(derive_group_key_and_drop_pii(raw, params), params)
weights = weight_kg(scoped["weight_kg"])

outlier_checks = checks(
    "Known data issues",
    [
        check("lowest price (EUR)", 1, raw["price"].min()),
        check("highest mileage_km_raw (km)", 2_570_000, raw["mileage_km_raw"].max()),
        check("scoped listings", 105_405, len(scoped)),
        check("lowest weight_kg, scoped (kg)", 1, weights.min()),
        check("highest weight_kg, scoped (kg)", 93_000, weights.max()),
        check(f"scoped rows below {LIGHT_KG} kg", 30, (weights < LIGHT_KG).sum()),
        check(f"scoped rows above {HEAVY_KG:,} kg", 9, (weights > HEAVY_KG).sum()),
        check("median weight_kg, scoped (kg)", 1_810, weights.median()),
    ],
)
outlier_checks

In [ ]:
SELLER_AND_LOCATION = ["country_code", "seller_type", "city", "zip", "street"]
no_country = raw["country_code"].isna()
without_any = raw[SELLER_AND_LOCATION].isna().all(axis=1)

location_checks = checks(
    "Known data issues",
    [
        check("rows with no seller or location data", 15, without_any.sum()),
        check("they are the rows with no country_code", True, (without_any == no_country).all()),
        check(
            "all of them carry coordinates",
            True,
            raw.loc[without_any, ["latitude", "longitude"]].notna().all().all(),
        ),
    ],
)
location_checks

## How this project uses the dataset

The scope, the row funnel, the `ES` holdout, the split and the supported makes.
These figures come from the pipeline rather than from the file alone, so they are recomputed with the `preprocess` and `split` stages' own functions and the parameters in `params.yaml`.
A change to either changes these numbers, and the card has to follow.

The `ES` count before deduplication comes from running the same rules on the `ES` rows alone.
The rules before deduplication look at one row at a time, so on the `ES` rows they keep exactly the `ES` rows the full run hands to deduplication.

In [ ]:
CARD_FUNNEL = [118_382, 110_013, 109_986, 109_911, 105_405]
CARD_REMOVED = {"registered by the reference date": 27, "deduplicated": 4_506}
rules = params["preprocess"]
offer_scope = raw["offer_type"] == rules["offer_type"]
vehicle_scope = raw["vehicle_type"] == rules["vehicle_type"]
price_range = f"{rules['price_min_eur']:,} to {rules['price_max_eur']:,}"
print(funnel.render())

funnel_rows = [
    check(
        "in scope before the pre-registered filter", 113_708, (offer_scope & vehicle_scope).sum()
    ),
    check("price range kept (EUR)", "500 to 2,000,000", price_range),
    check("post-snapshot listings in the raw file", 164, after_snapshot.sum()),
]
funnel_rows += [
    check(f"rows after: {step.rule}", rows, step.rows)
    for step, rows in zip(funnel.steps, CARD_FUNNEL, strict=True)
]
funnel_rows += [
    check(f"rows removed by: {rule}", rows, funnel.removed(rule))
    for rule, rows in CARD_REMOVED.items()
]
funnel_checks = checks("How this project uses the dataset", funnel_rows)
funnel_checks

In [ ]:
split_params = params["split"]
country, group_key = split_params["holdout_country"], split_params["group_key"]
in_country = raw[raw["country_code"] == country]
_, country_funnel = apply_row_rules(derive_group_key_and_drop_pii(in_country, params), params)
before_dedup = country_funnel.rows("price in the training range")
is_holdout = scoped["country_code"] == country
holdout, pool = scoped[is_holdout], scoped[~is_holdout]
group_sizes = pool[group_key].value_counts()
largest = group_sizes.iloc[0]
crossing = cross_holdout_sellers(holdout[group_key], pool[group_key])

holdout_checks = checks(
    "How this project uses the dataset",
    [
        check("holdout country", "ES", country),
        check("ES listings in the raw file", 8_015, len(in_country)),
        check("ES listings before deduplication", 7_545, before_dedup),
        check("ES holdout after preprocessing", 6_079, len(holdout)),
        check(
            "holdout share of the preprocessed rows (%)", 5.8, pct(len(holdout), len(scoped)), 1
        ),
        check("listings left to split", 99_326, len(pool)),
        check("seller groups among them", 28_435, len(group_sizes)),
        check("largest seller group", 3_383, largest),
        check("largest group's share of the pool (%)", 3.4, pct(largest, len(pool)), 1),
        check("sellers both in ES and outside it", 0, len(crossing)),
    ],
)
holdout_checks

In [ ]:
CARD_RATIOS = {"train": 0.60, "validation": 0.10, "calibration": 0.10, "test": 0.20}
CARD_SIZES = {
    "train": (61_180, 61.6),
    "validation": (8_992, 9.1),
    "calibration": (9_169, 9.2),
    "test": (19_985, 20.1),
}
assigned = assign_split(pool[group_key], split_params["ratios"], params["seed"])
sizes = {name: int((assigned == name).sum()) for name in SPLIT_NAMES}
shares = realised_shares(sizes)

split_rows = [
    check("seed", 20_251_108, params["seed"]),
    check("configured ratios", CARD_RATIOS, split_params["ratios"]),
]
for name, (size, share) in CARD_SIZES.items():
    split_rows.append(check(f"{name} listings", size, sizes[name]))
    split_rows.append(check(f"{name} share of the pool (%)", share, 100 * shares[name], 1))
split_checks = checks("How this project uses the dataset", split_rows)
split_checks

In [ ]:
CARD_SUPPORTED = {
    "BMW": 34_360,
    "Porsche": 22_434,
    "Mercedes-Benz": 13_444,
    "Audi": 12_115,
    "Alfa Romeo": 6_285,
    "Suzuki": 3_849,
    "Volvo": 3_348,
    "Honda": 798,
    "Hyundai": 548,
    "Aston Martin": 360,
    "Volkswagen": 348,
}
threshold = split_params["min_listings_per_make"]
supported = {row["make"]: row["listings"] for row in supported_makes(pool["make"], threshold)}
covered = sum(supported.values())
unfiltered = supported_makes(raw.loc[raw["country_code"] != country, "make"], threshold)
extra = sorted({entry["make"] for entry in unfiltered} - set(supported))

In [ ]:
make_scope_rows = [
    check("listings a make needs to be supported", 300, threshold),
    check("supported makes", 11, len(supported)),
    check("they are the card's makes, in its order", list(CARD_SUPPORTED), list(supported)),
]
make_scope_rows += [
    check(f"{make} listings in the pool", listings, supported.get(make, 0))
    for make, listings in CARD_SUPPORTED.items()
]
make_scope_rows += [
    check("listings of supported makes", 97_889, covered),
    check("their share of the pool (%)", 98.6, pct(covered, len(pool)), 1),
    check("supported makes on the unfiltered raw rows", 13, len(unfiltered)),
    check("the makes only the unfiltered rows support", "BYD, Ford", ", ".join(extra)),
]
make_scope_checks = checks("How this project uses the dataset", make_scope_rows)
make_scope_checks

## Personal and Sensitive Information

The section names the private listings whose seller location identifies a specific place.

In [ ]:
privacy_checks = checks(
    "Personal and Sensitive Information",
    [check("private listings", 19_802, (raw["seller_type"] == "PrivateSeller").sum())],
)
privacy_checks

## Bias, Risks, and Limitations

The brand, country and seller skews, and the rows that are not used passenger cars.
Absent makes are matched case- and accent-insensitively, so `Škoda` would be found as `Skoda` too.

In [ ]:
CARD_ABSENT_MAKES = ["Toyota", "SEAT", "Peugeot", "Fiat", "Škoda"]


def folded(name: str) -> str:
    """`name` without accents and case, so 'Škoda' and 'SKODA' compare equal."""
    return unicodedata.normalize("NFKD", name).encode("ascii", "ignore").decode().casefold()


present = {folded(make) for make in make_counts.index}
absent = [make for make in CARD_ABSENT_MAKES if folded(make) not in present]
brand_checks = checks(
    "Bias, Risks, and Limitations",
    [
        check("share of the four premium makes (%)", 82.9, premium_share, 1),
        check("Volkswagen listings", 352, make_counts.get("Volkswagen", 0)),
        check("Renault listings", 60, make_counts.get("Renault", 0)),
        check("Opel listings", 60, make_counts.get("Opel", 0)),
        check("makes the card names as absent that are absent", CARD_ABSENT_MAKES, absent),
    ],
)
brand_checks

In [ ]:
preregistered_offers = raw.loc[preregistered, "offer_type"]

skew_checks = checks(
    "Bias, Risks, and Limitations",
    [
        check("DE share (%)", 38.5, pct(countries["DE"], len(raw)), 1),
        check(
            "DE, IT and NL share (%)", 73.2, pct(countries[["DE", "IT", "NL"]].sum(), len(raw)), 1
        ),
        check("LU listings", 789, countries["LU"]),
        check("new listings (offer_type N)", 4_252, offers["N"]),
        check("pre-registered listings", 3_702, preregistered.sum()),
        check("all of them in the used bucket", True, (preregistered_offers == "U").all()),
        check("light commercial vehicles (Transporter)", 456, vehicles["Transporter"]),
        check("dealer share (%)", 83.3, pct(sellers["Dealer"], len(raw)), 1),
        check("distinct dealer company names", 17_141, raw["seller_company_name"].nunique()),
    ],
)
skew_checks

## Summary

Every check of every section in one table, in the card's order.

In [ ]:
summary = pd.concat(
    [
        details_checks,
        uses_checks,
        structure_checks,
        country_checks,
        make_checks,
        count_checks,
        range_checks,
        completeness_checks,
        issue_checks,
        duplicate_checks,
        leak_checks,
        redundancy_checks,
        outlier_checks,
        location_checks,
        funnel_checks,
        holdout_checks,
        split_checks,
        make_scope_checks,
        privacy_checks,
        brand_checks,
        skew_checks,
    ],
    ignore_index=True,
)
print(f"{int(summary['match'].sum())} of {len(summary)} card figures reproduce.")
display(summary[~summary["match"]])
assert summary["match"].all(), "a card figure does not reproduce; the table above lists it"
summary

## Conclusion

Every figure the card states and this notebook checks reproduces on the pinned file, and the summary cell fails when one does not, so `make notebook-run` checks the card as well as the code.
Writing this notebook corrected the card in three places.
The share of rows with the exact listing price in `description` is 6.8 %, matched as a whole number; a substring match, which also counts a price inside a longer number, gives 6.9 %.
`primary_fuel` overlaps `fuel_category` rather than refining it: 9 of its 14 levels occur under more than one fuel category.
There are no exact full-row duplicates only because `id` differs: 206 rows repeat an earlier row in every other column.
A figure that stops reproducing in a later run is a finding about the card: the fix belongs in the card, and the computation here stays as it is.